# 08 — Diagnóstico de la clasificación española

La clasificación del corpus español con mDeBERTa produjo resultados anómalos:
la categoría `charity` (estafa de donación / ONG) absorbió el 45% del corpus,
con ejemplos que en realidad son phishing/smishing.

Este notebook **NO reclasifica nada**. Solo analiza el CSV ya generado
(`scam_es_FINAL_classified.csv`) para entender la causa antes de decidir
una posible reejecución.

## Hipótesis a contrastar

1. La etiqueta de `charity` es demasiado genérica y atrae ruido.
2. La confianza de los tweets en `charity` es baja (el modelo "duda").
3. Los tweets de `charity` provienen mayoritariamente de la query `phishing`.
4. mDeBERTa reparte mal por las etiquetas largas en español.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

df = pd.read_csv("../data/raw/scam_es_FINAL_classified.csv")
print(f"Tweets: {len(df)}")
print(f"Columnas: {list(df.columns)}")


## Hipótesis 1 y 2 — Confianza por categoría

Si `charity` tiene confianza muy baja, confirma que el modelo lo usa como
"cajón de sastre" cuando no está seguro.


In [ ]:
conf_por_cat = df.groupby("predicted_category").agg(
    n=("confidence_score", "count"),
    conf_media=("confidence_score", "mean"),
    conf_mediana=("confidence_score", "median"),
    conf_min=("confidence_score", "min"),
    conf_max=("confidence_score", "max"),
).sort_values("n", ascending=False).round(3)

print("=== CONFIANZA POR CATEGORÍA ===\n")
print(conf_por_cat)
print()
charity_conf = df[df["predicted_category"] == "charity"]["confidence_score"]
otras_conf = df[df["predicted_category"] != "charity"]["confidence_score"]
print(f"Confianza media en 'charity':  {charity_conf.mean():.3f}")
print(f"Confianza media en el resto:   {otras_conf.mean():.3f}")


## Hipótesis 3 — ¿De qué query vienen los tweets de charity?

Si los tweets clasificados como `charity` venían de la query `phishing`,
confirma que son phishing mal clasificado.


In [ ]:
charity_df = df[df["predicted_category"] == "charity"]
print(f"Total en charity: {len(charity_df)}\n")

if "query_labels" in df.columns:
    print("=== QUERY DE ORIGEN de los tweets en 'charity' ===\n")
    # Contar apariciones de cada query label
    for q in ["phishing", "payment_apps", "crypto", "romance", "impersonation"]:
        n = charity_df["query_labels"].fillna("").str.contains(q).sum()
        pct = n / len(charity_df) * 100 if len(charity_df) else 0
        print(f"  {q:<16} {n:>5}  ({pct:.1f}%)")
    print()
    print("Comparación: distribución de queries en TODO el corpus:")
    for q in ["phishing", "payment_apps", "crypto", "romance", "impersonation"]:
        n = df["query_labels"].fillna("").str.contains(q).sum()
        pct = n / len(df) * 100
        print(f"  {q:<16} {n:>5}  ({pct:.1f}%)")
else:
    print("No hay columna query_labels")


## Hipótesis 4 — ¿Qué etiqueta exacta devolvió el modelo?

Revisamos la columna `predicted_label` (la etiqueta larga en español original).


In [ ]:
if "predicted_label" in df.columns:
    print("=== ETIQUETAS LARGAS devueltas para 'charity' ===\n")
    print(charity_df["predicted_label"].value_counts())
    print()
    print("=== TODAS las etiquetas largas del corpus ===\n")
    print(df["predicted_label"].value_counts())


## Muestra amplia de 'charity' para inspección visual

Vemos 30 tweets de charity con su confianza, ordenados de mayor a menor.


In [ ]:
print("=== 30 TWEETS CLASIFICADOS COMO 'charity' (mayor confianza) ===\n")
for _, row in charity_df.nlargest(30, "confidence_score").iterrows():
    q = row.get("query_labels", "?")
    print(f"[conf {row['confidence_score']:.2f}] [query: {q}]")
    print(f"  {str(row['text'])[:240]}")
    print()


In [ ]:
print("=== 15 TWEETS DE 'charity' CON MENOR CONFIANZA ===\n")
for _, row in charity_df.nsmallest(15, "confidence_score").iterrows():
    q = row.get("query_labels", "?")
    print(f"[conf {row['confidence_score']:.2f}] [query: {q}]")
    print(f"  {str(row['text'])[:240]}")
    print()


## Contraste — ¿cómo quedó phishing_identity?

phishing era el 74% del corpus crudo. Si quedó hundido, confirma el problema.


In [ ]:
phish_df = df[df["predicted_category"] == "phishing_identity"]
print(f"Tweets en phishing_identity: {len(phish_df)} ({len(phish_df)/len(df)*100:.1f}%)")
print()
print("Recordatorio: la query 'phishing' representaba ~74% del corpus crudo.")
print("Si phishing_identity quedó muy por debajo, el phishing real está")
print("disperso en otras categorías (probablemente charity).")
print()
if len(phish_df) > 0:
    print("Ejemplos de los que SÍ quedaron en phishing_identity:")
    for _, row in phish_df.nlargest(5, "confidence_score").iterrows():
        print(f"  [{row['confidence_score']:.2f}] {str(row['text'])[:200]}")


## Conclusión del diagnóstico

In [ ]:
print("=" * 60)
print("   RESUMEN DEL DIAGNÓSTICO")
print("=" * 60)

charity_conf_m = df[df['predicted_category']=='charity']['confidence_score'].mean()
charity_n = (df['predicted_category']=='charity').sum()
phish_n = (df['predicted_category']=='phishing_identity').sum()

# % de charity que venía de query phishing
if "query_labels" in df.columns:
    charity_from_phish = df[(df['predicted_category']=='charity')]['query_labels'].fillna('').str.contains('phishing').sum()
    pct_cfp = charity_from_phish / charity_n * 100 if charity_n else 0
else:
    pct_cfp = None

print(f"""
1. charity absorbió {charity_n} tweets ({charity_n/len(df)*100:.1f}% del corpus).
2. Confianza media en charity: {charity_conf_m:.3f}
   (corpus global: {df['confidence_score'].mean():.3f})
3. phishing_identity quedó en {phish_n} tweets ({phish_n/len(df)*100:.1f}%),
   pese a que la query phishing era ~74% del corpus crudo.
""")
if pct_cfp is not None:
    print(f"4. De los tweets en charity, el {pct_cfp:.1f}% venía de la query 'phishing'.")
print()
print("INTERPRETACIÓN:")
print("Si charity tiene confianza baja Y se nutre de la query phishing,")
print("la conclusión es que 'charity' actúa como cajón de sastre del modelo")
print("y la clasificación con mDeBERTa + etiquetas largas en español no es fiable.")
print("=" * 60)
